In [1]:
import pandas as pd


In [2]:
#시군구별 계약종별별 전력사용량	((단위 : kWh)) 불필요 열 제거
elect = pd.read_csv('./data/elect_2015_2024.csv',encoding='CP949')

# 강원도만 뽑기 
gangwon_elect = elect[elect['시도']=='강원도']
gangwon_elect['시군구'].unique()  # 18개 강원도 지역 포함

#기상청 데이터와 일치시키기 
gangwon_elect['지점명'] = gangwon_elect['시군구'].str.replace('시|군|구','',regex=True)
gangwon_elect = gangwon_elect.rename(columns={'연도':'years'})

# 기상청 데이터의 강원도 지역과 동일한 한국전력의 측정 지역 추출 (기상청 강원도 데이터의 지역수(9개 지역)가 더 적기 때문에 기준으로 진행)
gangwon_temp_list = ['속초', '철원', '춘천', '강릉', '동해', '원주', '영월', '인제', '홍천']

gangwon_elect_list = []
for i in gangwon_elect['시군구'].drop_duplicates():
    if i in gangwon_temp_list: # 기상청 강원도 리스트
        gangwon_elect_list.append(i)
gangwon_elect_list

# 강원도 지역의 속한 데이터프레임 뽑기 
gangwon_keyword = "|".join(gangwon_elect_list)
gangwon_elect = gangwon_elect[gangwon_elect['시군구'].str.contains(gangwon_keyword)].reset_index(drop=True)

# # 2번째 방법
# gangwon_elect = gangwon_elect[gangwon_elect['계약종별']=='합 계'].reset_index()
# gangwon_elect
contract_types = ['주택용', '일반용', '교육용', '산업용', '농사용', '가로등', '심 야', '합 계']

gangwon_dfs = {}

for ct in contract_types:
    gangwon_dfs[ct] = (
        gangwon_elect[gangwon_elect['계약종별'] == ct]
        .reset_index(drop=True)
    )
gangwon_house = gangwon_dfs['주택용']
gangwon_industry = gangwon_dfs['산업용']
gangwon_edu = gangwon_dfs['교육용']
gangwon_farm = gangwon_dfs['농사용']
gangwon_light = gangwon_dfs['가로등']
gangwon_night = gangwon_dfs['심 야']
gangwon_total = gangwon_dfs['합 계']


C:\Users\Playdata\AppData\Local\Temp\ipykernel_22508\1883548696.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gangwon_elect['지점명'] = gangwon_elect['시군구'].str.replace('시|군|구','',regex=True)


In [3]:
for contract_type, df in gangwon_dfs.items():
    melted_df = pd.melt(
        df,
        id_vars=['years', '시도', '시군구', '계약종별', '지점명'],
        value_vars=['1월','2월','3월','4월','5월','6월',
                    '7월','8월','9월','10월','11월','12월'],
        var_name='month',
        value_name='전력량'
    )

    melted_df['month'] = melted_df['month'].str.replace('월','', regex=True).astype(int)
    melted_df['전력량'] = melted_df['전력량'].astype(int)

    melted_df.to_csv(
        f'./data/preprocessing/contract_type/{contract_type}_elect_gangwon',
        index=False,
        encoding='CP949'
    )